In [ ]:
#| default_exp cli

# cli

This notebook defines `sic`, a JSON-first CLI for interacting with SolveIt dialogs.

## Imports

In [ ]:
#| export
from __future__ import annotations

from typing import Any

from fastcore.all import *
from fastcore.script import *
from solveit_client.core import *

import json, os, sys

In [ ]:
from contextlib import contextmanager, redirect_stdout
from io import StringIO

from fastcore.test import *

## Local Integration Setup

The checks below talk to a real SolveIt server on `http://localhost:6001` with a disposable dialog. `nbdev-test` should therefore be run in an environment where that local server is available.

In [ ]:
_CLI_TEST_URL = 'http://localhost:6001'
_CLI_TEST_TOKEN = 'stub'
_CLI_TEST_DLG = 'nbdev-cli-test'
_CLI_TEST_EXTRA_DLG = 'nbdev-cli-test-create'
_CLI_TEST_NOTE = '# Heading\nalpha note'
_CLI_TEST_CODE = 'x = 1\nprint(x)'


def _test_client(): return SolveItClient(_CLI_TEST_URL, token=_CLI_TEST_TOKEN)


def _cleanup_dialog(dlg_name:str):
    try: Dialog(dlg_name, _test_client()).delete()
    except Exception: pass


def _cleanup_cli_test_dialog(): _cleanup_dialog(_CLI_TEST_DLG)


def _prepare_cli_test_dialog():
    _cleanup_cli_test_dialog()
    dlg = _test_client().create_dialog(_CLI_TEST_DLG)
    heading = dlg.add_msg(_CLI_TEST_NOTE, msg_type='note')
    code = dlg.add_msg(_CLI_TEST_CODE, msg_type='code')
    return dlg, heading, code


@contextmanager
def _cli_env(url:str=_CLI_TEST_URL, token:str=_CLI_TEST_TOKEN, argv:list[str]|None=None, env:dict|None=None):
    old_env = {k: os.environ.get(k) for k in ('SOLVEIT_URL', 'SOLVEIT_TOKEN')}
    old_argv = list(sys.argv)
    try:
        vals = {'SOLVEIT_URL': url, 'SOLVEIT_TOKEN': token} | (env or {})
        for k,v in vals.items():
            if v is None: os.environ.pop(k, None)
            else: os.environ[k] = v
        if argv is not None: sys.argv = argv
        _STATE.client = None
        _STATE.pretty = False
        _STATE.url = ''
        _STATE.token = ''
        yield
    finally:
        sys.argv = old_argv
        for k,v in old_env.items():
            if v is None: os.environ.pop(k, None)
            else: os.environ[k] = v
        _STATE.client = None
        _STATE.pretty = False
        _STATE.url = ''
        _STATE.token = ''


def _run_cli(args:list[str], url:str=_CLI_TEST_URL, token:str=_CLI_TEST_TOKEN, env:dict|None=None):
    buf = StringIO()
    gargs, rest = _parser(_global_options, 'sic').parse_known_args(args)
    gargs = gargs.__dict__
    gargs.pop('pdb', None)
    gargs.pop('xtra', None)
    with _cli_env(url=url, token=token, env=env, argv=['sic', *rest]), redirect_stdout(buf):
        sic.__wrapped__(**gargs)
    txt = buf.getvalue().strip()
    return json.loads(txt) if txt else None


def _run_completion(words:list[str]):
    buf = StringIO()
    with _cli_env(argv=['completion-sic', str(len(words) - 1), *words]), redirect_stdout(buf):
        completion_sic()
    return buf.getvalue().strip().split()

## Runtime Helpers

The CLI keeps its runtime state deliberately small: command groups, field filters, and a small config object for the current URL, token, and pretty-print flag.

In [ ]:
#| export
_MSG_TYPES = ('code', 'note', 'prompt', 'raw')
_PLACEMENTS = ('add_after', 'add_before', 'at_start', 'at_end')
_DIALOG_FIELDS = ('name', 'link', 'mode', 'error')
_MESSAGE_FIELDS = ('id', 'dlg_name', 'link', 'msg_type', 'content', 'output', 'time_run', 'error')
_STATE = SimpleNamespace(client=None, pretty=False, url='', token='')


def _global_options(
    url:str='',  # SolveIt base URL. Defaults to `SOLVEIT_URL` or `http://localhost:5001`.
    token:str='',  # SolveIt token. Defaults to `SOLVEIT_TOKEN`.
    pretty:Param('Pretty-print JSON output', store_true)=False,
): ...

In [ ]:
#| export
def _serialize(o:Any):
    if isinstance(o, Dialog):
        data = dict(o.data or {})
        data['name'], data['link'] = o.name, o.link
        return {k: data[k] for k in _DIALOG_FIELDS if k in data}
    if isinstance(o, Message):
        data = dict(o.data or {})
        data['id'], data['dlg_name'], data['link'] = o.id, o.dlg.name, o.link
        return {k: data[k] for k in _MESSAGE_FIELDS if k in data}
    if isinstance(o, (Messages, list, tuple, L)): return [_serialize(x) for x in o]
    if isinstance(o, dict): return {k: _serialize(v) for k, v in o.items()}
    return o


def _dump(o:Any): return json.dumps(_serialize(o), sort_keys=True, indent=2 if _STATE.pretty else None)


def _parse_value(v:str):
    try: return json.loads(v)
    except json.JSONDecodeError: return v


def _parse_pairs(pairs:list[str]):
    res = {}
    for pair in pairs:
        if '=' not in pair: raise ValueError(f'Expected key=value pair: {pair}')
        k, v = pair.split('=', 1)
        res[k] = _parse_value(v)
    return res


def _norm_opt(s:str): return '--' + s[2:].replace('-', '_') if s.startswith('--') else s
def _disp_opt(s:str): return '--' + s[2:].replace('_', '-') if s.startswith('--') else s

In [ ]:
#| export
def _set_state(url:str='', token:str='', pretty:bool=False):
    _STATE.client = None
    _STATE.pretty = pretty
    _STATE.url = url
    _STATE.token = token


def _resolve_config(url:str='', token:str=''):
    url = url or _STATE.url or os.getenv('SOLVEIT_URL') or 'http://localhost:5001'
    token = token or _STATE.token or os.getenv('SOLVEIT_TOKEN', '')
    if not token: raise ValueError('Missing SolveIt token. Pass `--token` or set `SOLVEIT_TOKEN`.')
    return url, token


def _client():
    if _STATE.client is None:
        url, token = _resolve_config()
        _STATE.client = SolveItClient(url, token=token)
    return _STATE.client


def _dialog(dlg_name:str): return Dialog(dlg_name, _client())
def _message(dlg_name:str, msg_id:str): return _dialog(dlg_name).read_msg(id=msg_id)
def _emit(res:Any): print(_dump(res))

`_serialize` is intentionally selective. The goal is to remove runtime noise like event handles and queue state so both agents and humans get stable JSON back.

In [ ]:
_set_state(token=_CLI_TEST_TOKEN)
test_eq(_resolve_config()[0], 'http://localhost:5001')
_set_state()


## Dialog Commands

The dialog group is centered around the workflows that matter most for an agent: create, search, list, inspect as XML, and control execution state.

In [ ]:
#| export
def _dialog_create(
    dlg_name:str,  # Dialog name
):
    "Create a dialog."
    return _client().create_dialog(dlg_name)


def _dialog_delete(
    dlg_name:str,  # Dialog name
):
    "Delete a dialog."
    return _dialog(dlg_name).delete()


def _dialog_find(
    dlg_name:str,  # Dialog name
    re_pattern:str,  # Regex pattern to search for
    msg_type:Param('Optional message type filter', choices=_MSG_TYPES)=None,
):
    "Find messages in a dialog with a regex."
    kw = dict(msg_type=msg_type) if msg_type else {}
    return _dialog(dlg_name).find_msgs(re_pattern=re_pattern, **kw)


def _dialog_msgs(
    dlg_name:str,  # Dialog name
    msg_type:Param('Optional message type filter', choices=_MSG_TYPES)=None,
):
    "List dialog messages."
    return _dialog(dlg_name).find_msgs(**(dict(msg_type=msg_type) if msg_type else {}))


def _dialog_xml(
    dlg_name:str,  # Dialog name
    msg_type:Param('Optional message type filter', choices=_MSG_TYPES)=None,
    nums:bool=False,  # Include line numbers
    include_output:bool=False,  # Include output content
    trunc_out:bool=True,  # Truncate output in XML
    trunc_in:bool=False,  # Truncate input in XML
):
    "Render dialog messages as XML."
    return _dialog(dlg_name).to_xml(msg_type=msg_type, nums=nums, include_output=include_output, trunc_out=trunc_out, trunc_in=trunc_in)


def _dialog_stop(
    dlg_name:str,  # Dialog name
):
    "Stop the current run queue."
    return _dialog(dlg_name).stop()


def _dialog_reset(
    dlg_name:str,  # Dialog name
):
    "Reset dialog execution state."
    return _dialog(dlg_name).reset()


def _dialog_run_all(
    dlg_name:str,  # Dialog name
):
    "Queue all runnable messages."
    return _dialog(dlg_name).run_all()

## Message Commands

Message commands cover the main agent edit loop: add, read, update, execute, and make narrow text or line edits that return a diff.

In [ ]:
#| export
def _msg_add(
    dlg_name:str,  # Dialog name
    content:str,  # Message content
    msg_type:Param('Message type', choices=_MSG_TYPES)='code',
    placement:Param('Placement relative to `ref_id` or dialog', choices=_PLACEMENTS)='at_end',
    ref_id:str=None,  # Reference message id for relative placement
    heading_collapsed:int=0,  # Collapse heading section
    i_collapsed:int=0,  # Collapse input
    o_collapsed:int=0,  # Collapse output
):
    "Add a message."
    return _dialog(dlg_name).add_msg(content, msg_type=msg_type, placement=placement, id=ref_id,
        heading_collapsed=heading_collapsed, i_collapsed=i_collapsed, o_collapsed=o_collapsed)


def _msg_read(
    dlg_name:str,  # Dialog name
    msg_id:str=None,  # Message id
    n:int=0,  # Relative message offset when `msg_id` is not provided
):
    "Read a message by id or offset."
    return _dialog(dlg_name).read_msg(id=msg_id, n=n)


def _msg_exec(
    dlg_name:str,  # Dialog name
    msg_id:str,  # Message id
    timeout:int=30,  # Max seconds to wait
    poll_interval:float=0.2,  # Poll interval in seconds
):
    "Execute a message and return the refreshed message."
    msg = _message(dlg_name, msg_id)
    msg.exec(timeout=timeout, poll_interval=poll_interval)
    return msg


def _msg_update(
    dlg_name:str,  # Dialog name
    msg_id:str,  # Message id
    content:str=None,  # Updated content
    output:str=None,  # Updated output
    msg_type:Param('Updated message type', choices=_MSG_TYPES)=None,
):
    "Update a message."
    kw = {k: v for k, v in dict(content=content, output=output, msg_type=msg_type).items() if v is not None}
    if not kw: raise ValueError('Provide at least one of `--content`, `--output`, or `--msg-type`.')
    msg, diff = _message(dlg_name, msg_id).update(**kw)
    return {'message': msg, 'diff': diff}


def _msg_delete(
    dlg_name:str,  # Dialog name
    msg_id:str,  # Message id
):
    "Delete a message."
    _message(dlg_name, msg_id).delete()
    return dict(success=True, id=msg_id, dlg_name=dlg_name)

In [ ]:
#| export
def _msg_str_replace(
    dlg_name:str,  # Dialog name
    msg_id:str,  # Message id
    old_str:str,  # Exact string to replace
    new_str:str,  # Replacement string
):
    "Replace a unique string in a message."
    msg, diff = _message(dlg_name, msg_id).str_replace(old_str, new_str)
    return {'message': msg, 'diff': diff}


def _msg_insert_line(
    dlg_name:str,  # Dialog name
    msg_id:str,  # Message id
    insert_line:int,  # Zero-based insertion index
    new_str:str,  # New line content
):
    "Insert a line into a message."
    msg, diff = _message(dlg_name, msg_id).insert_line(insert_line, new_str)
    return {'message': msg, 'diff': diff}


def _msg_replace_lines(
    dlg_name:str,  # Dialog name
    msg_id:str,  # Message id
    start_line:int,  # One-based start line
    new_content:str,  # Replacement block
    end_line:int=None,  # Optional inclusive end line
):
    "Replace a line range in a message."
    msg, diff = _message(dlg_name, msg_id).replace_lines(start_line, end_line=end_line, new_content=new_content)
    return {'message': msg, 'diff': diff}


def _msg_del_lines(
    dlg_name:str,  # Dialog name
    msg_id:str,  # Message id
    start_line:int,  # One-based start line
    end_line:int=None,  # Optional inclusive end line
):
    "Delete a line range in a message."
    msg, diff = _message(dlg_name, msg_id).del_lines(start_line, end_line=end_line)
    return {'message': msg, 'diff': diff}

## Dispatch and Completion

The dispatch layer stays small on purpose. It normalizes dashed flags, routes to the dialog or message subcommands, and exposes Bash completion for command names, flags, and fixed choices.

In [ ]:
#| export
def _raw(
    path:str,  # SolveIt route, e.g. `/find_msgs_`
    pairs:Param('Request data in key=value form', nargs='*', opt=False)=(),
):
    "Call a raw SolveIt route."
    return _client()(path, **_parse_pairs(list(pairs)))


_DIALOG_CMDS = {'create': _dialog_create, 'delete': _dialog_delete, 'find': _dialog_find,
    'msgs': _dialog_msgs, 'xml': _dialog_xml, 'stop': _dialog_stop, 'reset': _dialog_reset, 'run-all': _dialog_run_all}

_MSG_CMDS = dict(add=_msg_add, read=_msg_read, exec=_msg_exec, update=_msg_update, delete=_msg_delete) | {
    'str-replace': _msg_str_replace, 'insert-line': _msg_insert_line, 'replace-lines': _msg_replace_lines, 'del-lines': _msg_del_lines}

_GROUPS = {'dialog': _DIALOG_CMDS, 'msg': _MSG_CMDS}


def _usage(): return 'Usage: sic [--url URL] [--token TOKEN] [--pretty] <dialog|msg|raw> ...'


def _group_usage(name:str, cmds:dict):
    joined = '|'.join(cmds)
    return f'Usage: sic {name} <{joined}>'


def _parser(func, prog:str): return anno_parser(func, prog=prog)


def _call(func, argv:list[str], prog:str):
    args = _parser(func, prog).parse_args([_norm_opt(o) for o in argv]).__dict__
    args.pop('pdb', None)
    args.pop('xtra', None)
    return func(**args)


def _dispatch(group:str, argv:list[str]):
    cmds = _GROUPS[group]
    if not argv:
        print(_group_usage(group, cmds))
        return None
    cmd, rest = argv[0], argv[1:]
    if cmd not in cmds: raise ValueError(f'Unknown {group} command: {cmd}')
    return _call(cmds[cmd], rest, f'sic {group} {cmd}')

In [ ]:
#| export
def _option_strings(parser): return [_disp_opt(o) for a in parser._actions for o in a.option_strings if o not in ('--pdb', '--xtra')]


def _choices_for(parser, opt:str):
    opt = _norm_opt(opt)
    for action in parser._actions:
        if opt in action.option_strings and action.choices: return [str(o) for o in action.choices]
    return []


def _filter(opts:list[str], prefix:str):
    seen = []
    for o in opts:
        if o.startswith(prefix) and o not in seen: seen.append(o)
    return seen


def _complete(words:list[str]):
    current = words[-1] if words else ''
    prev = words[-2] if len(words) > 1 else ''
    global_opts = _option_strings(_parser(_global_options, 'sic'))
    if not words: return list(_GROUPS) + ['raw', *global_opts]
    head = words[0]
    if len(words) == 1 and head not in _GROUPS and head != 'raw' and not head.startswith('-'):
        return _filter([*list(_GROUPS), 'raw', *global_opts], head)
    if head.startswith('-'): return _filter(global_opts, current)
    if head in _GROUPS and len(words) == 2 and words[1] and not words[1].startswith('-'):
        return _filter(list(_GROUPS[head]), words[1])
    if head in _GROUPS and len(words) == 2: return _filter(list(_GROUPS[head]), current)
    if head in _GROUPS:
        cmd = words[1]
        if cmd not in _GROUPS[head]: return _filter(list(_GROUPS[head]), cmd)
        parser = _parser(_GROUPS[head][cmd], f'sic {head} {cmd}')
    elif head == 'raw': parser = _parser(_raw, 'sic raw')
    else: return []
    choices = _choices_for(parser, prev)
    if choices: return _filter(choices, current)
    return _filter(_option_strings(parser), current)


_TAB_COMPLETION = """
_do_sic_completions()
{
    local IFS=$'\\n'
    COMPREPLY=($(completion-sic "$COMP_CWORD" "${COMP_WORDS[@]:1}"))
}

complete -F _do_sic_completions sic
"""

## Entry Points

In [ ]:
#| export
@call_parse(nested=True)
def sic(
    url:str='',  # SolveIt base URL. Defaults to `SOLVEIT_URL` or `http://localhost:5001`.
    token:str='',  # SolveIt token. Defaults to `SOLVEIT_TOKEN`.
    pretty:bool=False,  # Pretty-print JSON output
):
    "Python backend for the `sic` command."
    try:
        _set_state(url=url, token=token, pretty=pretty)
        rest = sys.argv[1:]
        if not rest:
            print(_usage())
            return
        head, rest = rest[0], rest[1:]
        if head in _GROUPS: res = _dispatch(head, rest)
        elif head == 'raw': res = _call(_raw, rest, 'sic raw')
        else: raise ValueError(f'Unknown command: {head}')
        if res is not None: _emit(res)
    except Exception as e: raise SystemExit(str(e)) from e


def completion_sic():
    "Python backend for `completion-sic`."
    if len(sys.argv) == 2 and sys.argv[1] == '--install':
        print(_TAB_COMPLETION)
        return
    cword = int(sys.argv[1]) if len(sys.argv) > 1 else 0
    words = sys.argv[2:]
    while len(words) <= cword: words.append('')
    print(' '.join(_complete(words[:cword + 1])))

## Live Checks

These checks exercise the exported CLI functions against a real disposable dialog on localhost. Cleanup runs in a `finally` block so the test dialog is removed even if one of the assertions fails.

In [ ]:
def _run_live_cli_checks():
    _cleanup_dialog(_CLI_TEST_EXTRA_DLG)
    _cleanup_cli_test_dialog()
    try:
        dlg, heading, code = _prepare_cli_test_dialog()

        res = _run_cli(['dialog', 'create', _CLI_TEST_EXTRA_DLG])
        test_eq(res['name'], _CLI_TEST_EXTRA_DLG)
        test_eq(res['mode'], 'learning')

        res = _run_cli(['dialog', 'find', _CLI_TEST_DLG, '^# '])
        test_eq([o['id'] for o in res], [heading.id])
        assert 'idle_evt' not in res[0]

        res = _run_cli(['dialog', 'msgs', _CLI_TEST_DLG])
        test_eq([o['id'] for o in res[:2]], [heading.id, code.id])

        res = _run_cli(['dialog', 'xml', _CLI_TEST_DLG, '--msg-type', 'note'])
        assert '<' in res and 'Heading' in res

        res = _run_cli(['msg', 'read', _CLI_TEST_DLG, '--msg-id', code.id])
        test_eq(res['content'], _CLI_TEST_CODE)

        res = _run_cli(['msg', 'add', _CLI_TEST_DLG, 'agent note', '--msg-type', 'note'])
        note_id = res['id']
        test_eq(res['content'], 'agent note')

        res = _run_cli(['msg', 'str-replace', _CLI_TEST_DLG, note_id, 'agent', 'tested'])
        test_eq(res['message']['content'], 'tested note')

        res = _run_cli(['msg', 'insert-line', _CLI_TEST_DLG, code.id, '1', 'y = 2'])
        test_eq(res['message']['content'], 'x = 1\ny = 2\nprint(x)')

        res = _run_cli(['msg', 'replace-lines', _CLI_TEST_DLG, code.id, '3', 'print(x + y)'])
        test_eq(res['message']['content'], 'x = 1\ny = 2\nprint(x + y)')

        res = _run_cli(['msg', 'exec', _CLI_TEST_DLG, code.id])
        test_eq(res['output'], '3')

        res = _run_cli(['raw', '/test_route'])
        test_eq(res, 'here')

        test_eq(_complete(['dialog', 'f']), ['find'])
        assert '--msg-type' in _complete(['msg', 'add', _CLI_TEST_DLG, 'print(1)', '--m'])
        test_eq(_run_completion(['dialog', 'f']), ['find'])

        try:
            _run_cli(['dialog', 'msgs', 'definitely-missing-dialog'])
            raise AssertionError('Expected a missing dialog failure')
        except SystemExit as e:
            assert str(e) == 'Dialog not found: definitely-missing-dialog'
    finally:
        _cleanup_dialog(_CLI_TEST_EXTRA_DLG)
        _cleanup_cli_test_dialog()

In [ ]:
_run_live_cli_checks()

## Usage

A typical agent loop is:

1. Discover the right place with `dialog find` or `dialog msgs`.
2. Read the target message with `msg read`.
3. Make one narrow edit with `msg str-replace`, `msg insert-line`, `msg replace-lines`, or `msg del-lines`.
4. Verify immediately with `msg exec` or another `msg read`.

Examples:

```bash
sic dialog find CRAFT '^# '
sic msg read CRAFT --msg-id _733cf1be
sic msg replace-lines CRAFT _733cf1be 3 'print(x + y)'
sic msg exec CRAFT _733cf1be
eval "$(completion-sic --install)"
```

`sic` defaults to `http://localhost:5001` when no URL is supplied. For local integration work in this notebook we use `http://localhost:6001` explicitly via the test helpers above.

In [ ]:
_cleanup_cli_test_dialog()